# Caro AlphaZero — Kaggle Training

**Setup:** Settings → Accelerator → GPU T4 x2

**Files cần upload:** `config.py`, `model_fcn.py`, `caro_agent_az.py`, `caro_env.py`, `greedy_agent.py`

In [ ]:
import os, time
import numpy as np
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {len(gpus)}')
for g in gpus: print(f'  {g}')

In [ ]:
from config import *
from caro_env import CaroEnv
from caro_agent_az import CaroAgent
from greedy_agent import GreedyAgent

CKPT_DIR = 'checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

---
## Bước 1: Pretrain từ Greedy


In [ ]:
agent_15  = CaroAgent(
    board_size=15, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE
)
greedy_15 = GreedyAgent(board_size=15)
env_15    = CaroEnv(board_size=15)

print(f'Params: {agent_15.model.count_params():,}')

In [ ]:
t0 = time.time()
agent_15.pretrain_from_greedy(
    greedy_15,
    n_games        = 7000,
    batch_size     = 64,
    temperature    = 1.2,
    random_opening = 2,
    skip_first     = 2,
)
print(f'Pretrain: {time.time()-t0:.0f}s')

In [ ]:
env_15.play_n(agent_15, greedy_15, n=20)

In [ ]:
agent_15.save(os.path.join(CKPT_DIR, 'pretrain_15.keras'))

---
## Bước 2: Train MCTS 15×15 với Replay Buffer

In [ ]:
agent_15_2 = CaroAgent(
    board_size=15, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE
)
agent_15_2.copy_weights_from(agent_15)

t0 = time.time()
env_15.train_with_buffer(
    agent_15, agent_15_2,
    n           = 2000,
    train_every = 20,
    batch_size  = 256,
    epochs      = 3,
    ckpt_every  = 500,
    plot_every  = 100,
)
print(f'Train 15x15: {time.time()-t0:.0f}s')

In [ ]:
agent_15.save(os.path.join(CKPT_DIR, 'train_15.keras'))
env_15.play_n(agent_15, greedy_15, n=20)

---
## Bước 3: Curriculum → 25×25

In [ ]:
agent_25  = CaroAgent(
    board_size=25, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE
)

agent_25_2 = CaroAgent(
    board_size=25, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE
)

greedy_25 = GreedyAgent(board_size=25)
env_25    = CaroEnv(board_size=25)

agent_25.copy_weights_from(agent_15)
agent_25_2.copy_weights_from(agent_15)

In [ ]:
t0 = time.time()
env_25.train_with_buffer(
    agent_25, agent_25_2,
    n           = 1000,
    train_every = 20,
    batch_size  = 256,
    epochs      = 3,
    ckpt_every  = 250,
    plot_every  = 50,
)
print(f'Train 25x25: {time.time()-t0:.0f}s')

In [ ]:
agent_25.save(os.path.join(CKPT_DIR, 'train_25.keras'))
env_25.play_n(agent_25, greedy_25, n=20)

---
## Bước 4: Curriculum → 40×40

In [ ]:
agent_40  = CaroAgent(
    board_size=40, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE
)

agent_40_2 = CaroAgent(
    board_size=40, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE
)

greedy_40 = GreedyAgent(board_size=40)
env_40    = CaroEnv(board_size=40)

agent_40.copy_weights_from(agent_25)
agent_40_2.copy_weights_from(agent_25)

In [ ]:
t0 = time.time()
env_40.train_with_buffer(
    agent_40, agent_40_2,
    n           = 600,
    train_every = 20,
    batch_size  = 256,
    epochs      = 3,
    ckpt_every  = 200,
    plot_every  = 50,
)
print(f'Train 40x40: {time.time()-t0:.0f}s')

In [ ]:
agent_40.save(os.path.join(CKPT_DIR, 'final_40.keras'))
env_40.play_n(agent_40, greedy_40, n=20)

---
## Xem kết quả

In [ ]:
env_40.play_one(agent_40, greedy_40, title='Agent vs Greedy')

In [ ]:
for f in sorted(os.listdir(CKPT_DIR)):
    size = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

---
## Load model đã lưu

In [ ]:
# agent = CaroAgent(board_size=40, base_filters=64, n_res=5, n_sim=50)
# agent.load('checkpoints/final_40.keras')

---
## Bảng tham số

| Tham số | Ở đâu | Ý nghĩa |
|---------|-------|----------|
| `base_filters`, `n_res` | `CaroAgent(...)` | Kích thước model (giống khi transfer) |
| `n_sim` | `CaroAgent(...)` | Số simulations MCTS |
| `n_parallel` | `CaroAgent(...)` | Số sim song song trên GPU |
| `buffer_size` | `CaroAgent(...)` | Buffer tối đa bao nhiêu positions |
| `n` | `env.train_with_buffer(...)` | Tổng số ván |
| `train_every` | `env.train_with_buffer(...)` | Chơi bao nhiêu ván rồi train |
| `batch_size` | `env.train_with_buffer(...)` | Batch size khi train |
| `epochs` | `env.train_with_buffer(...)` | Số epochs mỗi lần train |